# **Exploring LSOA characteristics for Mental Health related calls made to Police**

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
import folium
from folium.plugins import TimestampedGeoJson
from branca.colormap import linear

import geopandas as gpd
import plotly.graph_objs as go
import contextily
import json

from sklearn.linear_model import LinearRegression

import warnings

warnings.filterwarnings("ignore")

In [ ]:
sns.set(rc={'figure.figsize':(11, 4)})

In [ ]:
# Import combined Police dataset from 2017-22
df_import = pd.read_csv('BD_Policedata_2017_22_all.csv')
df_import = df_import.drop(df_import.columns[0],axis=1)
df = df_import.copy()
df['incident_start_date'] =pd.to_datetime(df['incident_start_date'])
df['incident_end_date'] =pd.to_datetime(df['incident_end_date'])

In [ ]:
# Import LSOA boundaries for Bradford
df_lsoa = gpd.read_file('LSOA_2011_full.geojson')
df_lsoa_bradford = df_lsoa[df_lsoa.LSOA11NM.str.contains("Bradford")]

In [ ]:
# Import population wrt 2021 census
df_pop = pd.read_csv('Population_bradford_21.csv')
df_pop = df_pop.dropna()

In [ ]:
df_lsoa_bradford[~df_lsoa_bradford.LSOA11CD.isin(df_pop['mnemonic'])]

In [ ]:
df_pop[~df_pop['mnemonic'].isin(df_lsoa_bradford.LSOA11CD)]

In [ ]:
df_pop.loc[df_pop['mnemonic'].str.contains('338'),'mnemonic'] = 'E01010677'
df_pop.loc[df_pop['mnemonic'].str.contains('339'),'mnemonic'] = 'E01010835'

In [ ]:
df_pop_21_11 = df_pop.groupby(['mnemonic'])['All persons'].sum().reset_index(name='Pop_21')
df_lsoa_bradford_pop = df_lsoa_bradford.merge(df_pop_21_11,left_on='LSOA11CD',right_on='mnemonic',how='inner')
df_lsoa_bradford_pop.drop('mnemonic',axis=1,inplace=True)

In [ ]:
# Combine two datasets based on LSOA information
merged_df = pd.merge(df_lsoa_bradford_pop, df, how='inner', left_on='LSOA11CD', right_on='lsoa')
merged_df.plot()

In [ ]:
lsoa_mh_df = merged_df.groupby('LSOA11CD').agg({'YearTrunk':'count','Pop_21':'mean'}).reset_index()
lsoa_mh_df['MH_call_rate']=lsoa_mh_df.YearTrunk/lsoa_mh_df.Pop_21

In [ ]:
lsoa_mh_df.rename(columns={'YearTrunk':'Total_MH_calls'},inplace=True)

In [ ]:
#Plot top 20 LSOAs
lsoa_mh_df.sort_values('MH_call_rate', ascending=True)[-20:].plot(
    kind='barh',
    color='blue',
    x='LSOA11CD',  # Replace with your LSOA column name if different
    y='MH_call_rate',
    figsize=(10, 6),
    title='MH Call Rate by LSOA',
    xlabel='MH Call Rate',
    ylabel='LSOA',
    legend=False
)

In [ ]:
lsoa_crime = merged_df.groupby(['LSOA11CD','CrimeTag']).size().reset_index(name='count').pivot_table(index='LSOA11CD',columns='CrimeTag',values='count',fill_value=0).reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_crime,on='LSOA11CD')
lsoa_mh_df['%non-crime'] = 100*(lsoa_mh_df['Non-Crime']/lsoa_mh_df.Total_MH_calls)

In [ ]:
#Plot top 20 LSOAs
lsoa_mh_df.sort_values('%non-crime', ascending=True)[-20:].plot(
    kind='barh',
    color='blue',
    x='LSOA11CD',  # Replace with your LSOA column name if different
    y='%non-crime',
    figsize=(10, 6),
    title='% of Non-Crime related calls by LSOA',
    xlabel='% of Non-Crime incidents',
    ylabel='LSOA',
    legend=False
)

In [ ]:
bins = [0,6,22,23]
labels = ['12AM-6AM','6AM-10PM','10PM-12AM']
merged_df['TimeBin'] = pd.cut(merged_df['incident_start_hour'], bins, labels=labels, right=False)
lsoa_night_mh = merged_df.groupby(['LSOA11CD','TimeBin']).size().reset_index(name='count').pivot_table(index='LSOA11CD',columns='TimeBin',values='count',fill_value=0).reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_night_mh,on='LSOA11CD')
lsoa_mh_df['Night-time_ratio'] = (lsoa_mh_df['12AM-6AM']+lsoa_mh_df['10PM-12AM'])/lsoa_mh_df.Total_MH_calls

In [ ]:
#Plot top 20 LSOAs
lsoa_mh_df.sort_values('Night-time_ratio', ascending=True)[-20:].plot(
    kind='barh',
    color='blue',
    x='LSOA11CD',  # Replace with your LSOA column name if different
    y='Night-time_ratio',
    figsize=(10, 6),
    title='Ratio of Night-time calls by LSOA',
    xlabel='Ratio of Night-time(10PM to 6AM) incidents',
    ylabel='LSOA',
    legend=False
)

In [ ]:
# Estimate temporal volatility (coefficient of variation) considering FY
merged_df['incident_start_month_num']=merged_df['incident_start_date'].dt.month
lsoa_monthly_calls = merged_df.groupby(['LSOA11CD', 'YearTrunk','incident_start_month_num']).size().reset_index(name='Monthly_call_volume')
lsoa_monthly_calls_summ = lsoa_monthly_calls.groupby('LSOA11CD')['Monthly_call_volume'].agg(['mean','std']).reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_monthly_calls_summ,on='LSOA11CD')
lsoa_mh_df['Temporalvolatility(CV)'] = lsoa_mh_df['std']/lsoa_mh_df['mean']
lsoa_mh_df

In [ ]:
lsoa_mh_df.sort_values('Temporalvolatility(CV)', ascending=True)[-20:].plot(
    kind='barh',
    color='blue',
    x='LSOA11CD',  # Replace with your LSOA column name if different
    y='Temporalvolatility(CV)',
    figsize=(10, 6),
    title='Temporal volatility by LSOA',
    xlabel='Coefficient of Variation for monthly calls',
    ylabel='LSOA',
    legend=False
)

In [ ]:
# Estimate temporal volatility (coefficient of variation) without FY
# lsoa_monthly_calls_2 = merged_df.groupby(['LSOA11CD', merged_df['incident_start_date'].dt.to_period('M')]).size().reset_index(name='Monthly_call_volume')
# lsoa_monthly_calls_summ_2 = lsoa_monthly_calls_2.groupby('LSOA11CD')['Monthly_call_volume'].agg(['mean','std']).reset_index()

# lsoa_mh_df.sort_values('Temporalvolatility(CV)', ascending=True)[-20:].plot(
#     kind='barh',
#     color='blue',
#     x='LSOA11CD',  # Replace with your LSOA column name if different
#     y='Temporalvolatility(CV)',
#     figsize=(10, 6),
#     title='Temporal volatility by LSOA',
#     xlabel='Coefficient of Variation for monthly calls',
#     ylabel='LSOA',
#     legend=False
# )

In [ ]:
lsoa_attended = merged_df.groupby(['LSOA11CD','attended']).size().reset_index(name='count').pivot_table(index='LSOA11CD',columns='attended',values='count',fill_value=0).reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_attended,on='LSOA11CD')
lsoa_mh_df['%attended'] = lsoa_mh_df[True]/lsoa_mh_df['Total_MH_calls']

In [ ]:
lsoa_grade = merged_df.groupby(['LSOA11CD','final_grade']).size().reset_index(name='count').pivot_table(index='LSOA11CD',columns='final_grade',values='count',fill_value=0).reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_grade,on='LSOA11CD')
lsoa_mh_df['%highacuity'] = (lsoa_mh_df['EMERGENCY']+lsoa_mh_df['PRIORITY'])/lsoa_mh_df['Total_MH_calls']

In [ ]:
# Estimating linear trend for monthly call volumes for each LSOAs
lsoa_monthly_trend = merged_df.groupby(['LSOA11CD', merged_df['incident_start_date'].dt.to_period('M')]).size().reset_index(name='Monthly_call_volume')
lsoa_monthly_trend['incident_start_date']= lsoa_monthly_trend['incident_start_date'].astype(str)
lsoa_monthly_trend['incident_start_date'] = pd.to_datetime(lsoa_monthly_trend['incident_start_date'])
lsoa_monthly_trend.sort_values(['LSOA11CD', 'incident_start_date'],inplace=True)
lsoa_monthly_trend['time_index'] = (
    lsoa_monthly_trend
    .groupby('LSOA11CD')['incident_start_date']
    .transform(lambda x: (x - x.min()).dt.days // 30)
)

In [ ]:
results = []

# Group by LSOA and fit a linear model
for lsoa, group in lsoa_monthly_trend.groupby('LSOA11CD'):
    if len(group) >= 3:  # Minimum data points to fit a trend
        X = group[['time_index']].values
        y = group[['Monthly_call_volume']].values

        model = LinearRegression()
        model.fit(X, y)

        slope = model.coef_[0][0]
        intercept = model.intercept_[0]        
    else:
        slope = 0
        intercept =0
    results.append({'LSOA11CD': lsoa, 'slope': slope, 'intercept': intercept})

In [ ]:
lsoa_linear_trend = pd.DataFrame(results)
lsoa_mh_df = lsoa_mh_df.merge(lsoa_linear_trend,on='LSOA11CD')

In [ ]:
lsoa_mh_df[lsoa_mh_df['slope']>=0]

In [ ]:
lsoa_mh_df.sort_values('slope', ascending=False)

In [ ]:
#Estimating Location Quotient

lsoa_call_lq = merged_df.groupby(['LSOA11CD','YearTrunk']).size().reset_index(name='Yearly_calls')
lsoa_call_lq_pivot = lsoa_call_lq.pivot_table(index='LSOA11CD',columns='YearTrunk',values='Yearly_calls',fill_value=0)
lsoa_call_lq_pivot = lsoa_call_lq_pivot.apply(lambda col: col/col.sum(),axis=0)
lsoa_call_lq_pivot['lsoa_mean'] = lsoa_call_lq_pivot.mean(axis=1)
lsoa_call_lq_pivot = lsoa_call_lq_pivot.reset_index()
lsoa_mh_df = lsoa_mh_df.merge(lsoa_call_lq_pivot[['LSOA11CD','lsoa_mean']],on='LSOA11CD')

In [ ]:
lsoa_mh_df['lsoa_tot_share'] = lsoa_mh_df['Total_MH_calls']/lsoa_mh_df['Total_MH_calls'].sum()
lsoa_mh_df['lsoa_lq'] = lsoa_mh_df['lsoa_mean']/(lsoa_mh_df['Pop_21']/lsoa_mh_df['Pop_21'].sum())

In [ ]:
lsoa_monthly_calls['Relative_change'] = lsoa_monthly_calls['Monthly_call_volume'].pct_change().fillna(0)
lsoa_percent_change = lsoa_monthly_calls.groupby('LSOA11CD')['Relative_change'].mean().reset_index(name='Percentage change')
lsoa_mh_df = lsoa_mh_df.merge(lsoa_percent_change,on='LSOA11CD')

In [ ]:
lsoa_mh_df = lsoa_mh_df.merge(df_lsoa_bradford_pop[['LSOA11CD','geometry']],on='LSOA11CD')
lsoa_mh_gdf = gpd.GeoDataFrame(lsoa_mh_df,geometry=lsoa_mh_df['geometry'])

In [ ]:
fig, ax = plt.subplots(1, figsize=(9, 9))
# Build map
lsoa_mh_gdf.plot(
    column='MH_call_rate',
    cmap="viridis",
    scheme="quantiles", #install mapclassify
    k=5,
    edgecolor="white",
    linewidth=0.0,
    alpha=0.75,
    legend=True,
    legend_kwds=dict(loc=2),
    ax=ax,
)
# Add basemap
contextily.add_basemap(
    ax,
    crs=lsoa_mh_gdf.crs,
    source=contextily.providers.CartoDB.VoyagerNoLabels,
)
# Remove axes
ax.set_axis_off()

In [ ]:
fig, ax = plt.subplots(1, figsize=(9, 9))
# Build map
lsoa_mh_gdf.plot(
    column='%non-crime',
    cmap="viridis",
    scheme='naturalbreaks', #install mapclassify
    k=5,
    edgecolor="white",
    linewidth=0.2,
    alpha=0.75,
    legend=True,
    legend_kwds=dict(loc=2),
    ax=ax,
)
# Add basemap
contextily.add_basemap(
    ax,
    crs=lsoa_mh_gdf.crs,
    source=contextily.providers.CartoDB.VoyagerNoLabels,
)
# Remove axes
ax.set_axis_off()

In [ ]:
fig, ax = plt.subplots(1, figsize=(9, 9))
# Build map
lsoa_mh_gdf.plot(
    column='Temporalvolatility(CV)',
    cmap="viridis",
    scheme='naturalbreaks', #install mapclassify
    k=5,
    edgecolor="white",
    linewidth=0.2,
    alpha=0.75,
    legend=True,
    legend_kwds=dict(loc=2),
    ax=ax,
)
# Add basemap
contextily.add_basemap(
    ax,
    crs=lsoa_mh_gdf.crs,
    source=contextily.providers.CartoDB.VoyagerNoLabels,
)
# Remove axes
ax.set_axis_off()

In [ ]:
fig, ax = plt.subplots(1, figsize=(9, 9))
# Build map
lsoa_mh_gdf.plot(
    column='lsoa_lq',
    cmap="viridis",
    scheme='naturalbreaks', #install mapclassify
    k=5,
    edgecolor="white",
    linewidth=0.2,
    alpha=0.75,
    legend=True,
    legend_kwds=dict(loc=2),
    ax=ax,
)
# Add basemap
contextily.add_basemap(
    ax,
    crs=lsoa_mh_gdf.crs,
    source=contextily.providers.CartoDB.VoyagerNoLabels,
)
# Remove axes
ax.set_axis_off()

In [ ]:
lsoa_mh_df.to_csv('Estimated_indicators_MH_Police.csv')